In [1]:
!pip install torch numpy matplotlib seaborn scikit-learn

In [3]:
"""
4차시 실습 통합 실행 파일
모든 실습을 한 번에 실행할 수 있습니다.

Part 1: 기본 설정 및 모델 정의
Part 2: 지도학습 (분류와 회귀)
Part 3: 비지도학습과 편향-분산
Part 4: K-Fold 교차검증
Part 5: 평가 지표 계산
Part 6: 전체 ML 파이프라인

필수 라이브러리:
pip install torch numpy matplotlib seaborn scikit-learn
"""

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_regression, make_blobs
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.linear_model import Ridge
from sklearn.cluster import KMeans

# https://rk1993.tistory.com/101
# https://velog.io/@jhlee508/%EB%A8%B8%EC%8B%A0%EB%9F%AC%EB%8B%9D-K-%ED%8F%89%EA%B7%A0K-Means-%EC%95%8C%EA%B3%A0%EB%A6%AC%EC%A6%98
# 재현성을 위한 시드 고정
torch.manual_seed(42)
np.random.seed(42)

# 한글 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False


print("=" * 70)
print("4차시 실습: 인공지능 개론 - 통합 실행")
print("=" * 70)

4차시 실습: 인공지능 개론 - 통합 실행


In [4]:
# 모델정의
# 이진 분류용 다층 퍼셉트론 (mlp: multi layer perceptron)
class BinaryClassifier(nn.Module):
    def __init__(self, input_dim):
        super(BinaryClassifier, self).__init__()
        self.layer1 = nn.Linear(input_dim, 64)
        self.layer2 = nn.Linear(64, 32)
        self.layer3 = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.dropout(x)
        x = self.relu(self.layer2(x))
        x = self.dropout(x)
        x = self.sigmoid(self.layer3(x))
        return x

In [5]:
class Regressor(nn.Module):
    # 회귀용 다층 퍼셉트론(mlp)
    def __init__(self, input_dim):
        super(Regressor, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

In [7]:
# 지도학습 : 분류와 회귀

# 분류 데이터 생성 및 학습
X_class,y_class =\
make_classification(n_samples=1000, n_features=20, n_informative=15,
                    weights=[0.7, 0.3], n_redundant=5, random_state=42)

데이터 분할

In [9]:
X_train_c, X_temp_c, y_train_c, y_temp_c =\
train_test_split(X_class, y_class, test_size=0.4, random_state=42, stratify=y_class)

In [10]:
X_val_c, X_test_c, y_val_c, y_test_c =\
train_test_split(X_temp_c, y_temp_c, test_size=0.5, random_state=42, stratify=y_temp_c)

In [13]:
print(X_train_c.shape, X_val_c.shape, X_test_c.shape)
print(y_train_c.shape, y_val_c.shape, y_test_c.shape)
print(X_train_c[:3])
print(y_test_c[:3])

(600, 20) (200, 20) (200, 20)
(600,) (200,) (200,)
[[ 3.25808892 12.33624211  1.74445683  0.73149444  2.66425414 -1.17302324
   4.22592432 -0.09568354  1.97419982 -4.34465788  3.50894079 -5.3115027
   0.08242234 -2.27609578  0.12520547 -6.69328022  1.40263083 -4.66711833
  -3.56061169 -1.85034528]
 [-0.29817082  7.03442924  4.19530174  3.46476416  0.52725002 -0.95091527
   2.57471833  0.17026885  3.2104434  -2.79241562  0.7826237  -1.96290982
   0.91302396 -3.90714019  2.06441333  1.03179495 -1.97156388  2.50924969
  -0.1938771   1.37985847]
 [-1.92647692  3.69638637  2.65013671  0.86456154 -0.64646315 -0.21724925
   2.54140029  2.20485676 -0.76591624 -1.08533698 -2.92221994  8.14830026
  -4.67835138 -1.86496132 -2.49945458  2.61197924 -4.5934369   0.45558078
   0.6459925   1.92962812]]
[1 0 0]


데이터 전처리

In [15]:
# 표준화 (x-x_bar) / sigma

scaler_c = StandardScaler() # 표준화
# 주의) train data 만 훈련(fit)함
X_train_c_scaled = scaler_c.fit_transform(X_train_c)
X_val_c_scaled = scaler_c.transform(X_val_c)
X_test_c_scaled = scaler_c.transform(X_test_c)

print(X_train_c_scaled[:3])
print(X_val_c_scaled[:3])
print(X_test_c_scaled[:3])

[[ 0.49748868  2.08786528  0.53680045  0.59480332  0.86073877 -0.29101822
   1.76290291 -0.2428268   0.85298761 -1.58391054  1.4897009  -1.05173387
   0.06406772 -0.53985204  0.0237404  -0.92309464  0.86413336 -0.60734588
  -1.32225908 -0.64840123]
 [-0.13015214  1.0186695   1.53185879  1.80625615  0.01681913 -0.19129263
   0.97998758 -0.12606372  1.34115877 -0.96384286  0.19494227 -0.41665515
   0.38962439 -1.12825839  0.83602132  0.52625525 -0.64717808  0.74394724
  -0.03714602  0.51926066]
 [-0.4175303   0.3454995   0.90451214  0.65378196 -0.44668937  0.13812047
   0.96418992  0.76719673 -0.22903671 -0.28192334 -1.56452961  1.50099047
  -1.80193111 -0.3915335  -1.07565763  0.82272356 -1.82152249  0.35724624
   0.28343981  0.71799272]]
[[ 0.18989849 -1.85672979 -0.43969795  0.64257527 -1.13269405  1.54263135
   1.25347063  0.44331416  0.61048    -0.76701655  0.85374835 -0.03155713
  -1.1086797   0.69803267 -1.50954254  0.61683334 -1.55271761  0.56570923
   1.33195619 -0.81732086]
 [-

In [21]:
# 데이터타입 변경 (torch.tensor)

X_train_c_t = torch.FloatTensor(X_train_c_scaled)
y_train_c_t = torch.FloatTensor(y_train_c).unsqueeze(1)
X_val_c_t = torch.FloatTensor(X_val_c_scaled)
y_val_c_t = torch.FloatTensor(y_val_c).unsqueeze(1)


model_class = BinaryClassifier(input_dim=20)
criterion = nn.BCELoss() # binary cross entropy loss
optimizer = optim.Adam(model_class.parameters(), lr=0.001)

In [18]:
print(model_class)

BinaryClassifier(
  (layer1): Linear(in_features=20, out_features=64, bias=True)
  (layer2): Linear(in_features=64, out_features=32, bias=True)
  (layer3): Linear(in_features=32, out_features=1, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
  (sigmoid): Sigmoid()
)


In [20]:
print(criterion)

BCELoss()


In [22]:
print(optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)
